# Analyse des ressources en eau en Tunisie

Ce notebook analyse les données sur l'eau en Tunisie pour identifier les régions à risque de pénurie.

## 1. Chargement des données

In [ ]:
import pandas as pd
import plotly.express as px
import folium

# Charger les données
df = pd.read_csv('water_data.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.dropna()
df.head()

## 2. Analyse des données

In [ ]:
# Consommation moyenne par région
avg_consumption = df.groupby('region')['consumption_m3'].mean()
print('Consommation moyenne par région (m³) :')
print(avg_consumption)

# Mois de pénurie (consommation supérieure à la disponibilité)
shortage_months = df[df['consumption_m3'] > df['availability_m3']].shape[0]
print(f'Nombre de mois avec pénurie : {shortage_months}')

## 3. Visualisations

In [ ]:
# Graphique de comparaison
fig = px.bar(df, x='date', y=['consumption_m3', 'availability_m3'],
             title='Consommation vs Disponibilité par mois',
             labels={'value': 'Volume (m³)', 'variable': 'Type', 'date': 'Mois'},
             barmode='group')
fig.show()

In [ ]:
# Carte : rouge si la consommation dépasse la disponibilité, vert sinon
m = folium.Map(location=[34.5, 10.0], zoom_start=6)
for _, row in df.iterrows():
    color = 'red' if row['consumption_m3'] > row['availability_m3'] else 'green'
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=8,
        color=color,
        fill=True,
        popup=f"{row['region']} : {row['consumption_m3']} m³ consommés, {row['availability_m3']} m³ disponibles"
    ).add_to(m)
m